# Contexte
Une entreprise possède plusieurs bâtiments équipés de capteurs IoT.
Chaque capteur collecte régulièrement des informations sur la température, l'humidité, la pression,
la consommation énergétique, le bâtiment, la date et l'heure de la mesure.
Chaque mesure possède également un état (OK, ALERTE et ERREUR).
L'objectif de l'atelier est de construire un modèle capable de prédire automatiquement l'état d'un
capteur à partir de ses mesures.
L'atelier suivra le workflow classique du Machine Learning :
Dataset → Chargement → Exploration → Nettoyage → X / y → Train / Test → Prétraitement →
Modèle → fit()→ predict()→ Évaluation → Sauvegarde → Chargement → Réutilisation

In [1]:
# ! pip install seaborn matplotlib pandas scikit-learn joblib

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer


In [3]:
df = pd.read_csv("../data/mesures_capteurs.csv")
df.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


# Partie 1 – Gestion des doublons

## 1) vérifier l’existence de doublons dans df

In [4]:
nb_doublons = df.duplicated().sum()
print(f"Nombre de doublons : {nb_doublons}")

Nombre de doublons : 5


## 2) le cas échéant, supprimer les doublons puis vérifier la suppression

In [5]:
# 2) Suppression des doublons (le cas échéant) puis vérification
if nb_doublons > 0:
    df = df.drop_duplicates().reset_index(drop=True)

print(f"Nombre de doublons après suppression : {df.duplicated().sum()}")
print("Nouvelles dimensions :", df.shape)


Nombre de doublons après suppression : 0
Nouvelles dimensions : (600, 9)


# Partie 2 – Sélection de y (cible) et X (caractéristiques)

## 1) Définir "etat" comme la cible ou valeur à prédire et "temperature", "humidite", "pression" et "consommation" comme caractéristiques ou variables explicatives

In [6]:
print("Valeurs manquantes de la cible 'etat' :", df["etat"].isna().sum())
df = df.dropna(subset=["etat"]).reset_index(drop=True)
print("Dimensions après retrait des cibles manquantes :", df.shape)


Valeurs manquantes de la cible 'etat' : 4
Dimensions après retrait des cibles manquantes : (596, 9)


In [7]:
features = ["temperature", "humidite", "pression", "consommation"]
target = "etat"

X = df[features]
y = df[target]

## 2) Afficher les cinq premières lignes de X et de y

In [8]:
X.head()

,temperature,humidite,pression,consommation
0,25.46,58.06,1008.95,287.28
1,24.00,79.73,993.39,116.20
2,25.82,54.47,1010.32,288.50
3,28.23,69.39,1019.62,136.65
4,20.58,53.80,1016.58,182.62


In [9]:
y.head()

0    OK
1    OK
2    OK
3    OK
4    OK
Name: etat, dtype: str

## 3) Quel est le type du problème de machine learning ?

Il s'agit d'un problème d'**apprentissage supervisé de classification multi-classes** : la cible
etat est une variable catégorielle à trois modalités (OK, ALERTE, ERREUR), et le modèle
doit apprendre, à partir d'exemples déjà étiquetés, à prédire la bonne classe pour de nouvelles
observations.

# Partie 3 – Découpage Train/Test

Divisons X en deux ensembles distincts : un pour l'entraînement (train) et un pour le test (test).
Avec les conditions suivantes : 20% des données serviront au test ; garantir la reproductibilité du
découpage ; conserver les mêmes proportions de classes dans l'ensemble de train et de test que
dans les données d'origine.

In [10]:
# 20% pour le test, reproductibilité (random_state) et conservation des proportions
# de classes (stratify=y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train :", X_train.shape, "| X_test :", X_test.shape)
print("\nProportions des classes (données d'origine) :")
print(y.value_counts(normalize=True))
print("\nProportions des classes (train) :")
print(y_train.value_counts(normalize=True))
print("\nProportions des classes (test) :")
print(y_test.value_counts(normalize=True))

X_train : (476, 4) | X_test : (120, 4)

Proportions des classes (données d'origine) :
etat
OK        0.942953
ALERTE    0.048658
ERREUR    0.008389
Name: proportion, dtype: float64

Proportions des classes (train) :
etat
OK        0.943277
ALERTE    0.048319
ERREUR    0.008403
Name: proportion, dtype: float64

Proportions des classes (test) :
etat
OK        0.941667
ALERTE    0.050000
ERREUR    0.008333
Name: proportion, dtype: float64


# Partie 4 – Gestion des valeurs manquantes

## 1) Vérifier l’existence de valeurs manquantes

In [11]:
print("Valeurs manquantes X_train :")
print(X_train.isna().sum())
print("\nValeurs manquantes X_test :")
print(X_test.isna().sum())

Valeurs manquantes X_train :
temperature     5
humidite        4
pression        5
consommation    3
dtype: int64

Valeurs manquantes X_test :
temperature     1
humidite        1
pression        0
consommation    2
dtype: int64


## 2) Sélectionner SimpleImputer avec la médiane

In [13]:
imputer = SimpleImputer(strategy="median")

## 3) Qu’est ce qui justifie le choix de la médiane ?

La médiane est plus **robuste aux valeurs extrêmes (outliers)** que la moyenne : une mesure
aberrante de température, pression ou consommation ne fait pas basculer la valeur de remplacement
comme le ferait la moyenne. Les capteurs IoT peuvent produire des mesures bruitées ou extrêmes lors
de pannes, ce qui justifie ce choix pour des variables numériques continues.
D’ailleurs, quand on a visualisé les données, on a observé des valeurs extrêmes. 


## 4) Trouver les paramètres (médianes) de l’imputeur sur X_train

In [14]:
imputer.fit(X_train)
print("Médianes apprises :", dict(zip(features, imputer.statistics_)))

Médianes apprises : {'temperature': np.float64(24.9), 'humidite': np.float64(65.38), 'pression': np.float64(1012.3), 'consommation': np.float64(206.59)}


## 5) Déterminer X_train_imputed et X_test_imputed, les transformés de X_train et X_test

In [15]:
X_train_imputed = pd.DataFrame(imputer.transform(X_train), columns=features, index=X_train.index)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=features, index=X_test.index)

print("Valeurs manquantes restantes (train) :", X_train_imputed.isna().sum().sum())
print("Valeurs manquantes restantes (test) :", X_test_imputed.isna().sum().sum())
X_train_imputed.head()

Valeurs manquantes restantes (train) : 0
Valeurs manquantes restantes (test) : 0


,temperature,humidite,pression,consommation
97,26.14,84.97,1003.59,142.31
473,22.27,44.03,1005.85,138.16
192,28.13,75.78,1003.24,242.62
246,25.51,68.57,1024.71,96.51
127,33.60,79.80,1021.96,212.55
